# recomendação visual de imagens

este notebook monta um sistema simples de recomendação por similaridade visual.

a ideia é usar uma rede pré-treinada para extrair características das imagens e comparar essas características com cosine similarity.

o projeto não treina uma rede do zero. o foco aqui é usar transfer learning para montar um fluxo pequeno e funcional.

## instalação das bibliotecas

no colab, normalmente tensorflow, numpy e matplotlib já vêm instalados. mesmo assim, esta célula garante as dependências principais do projeto.

In [ ]:
!pip install -q tensorflow numpy matplotlib scikit-learn opencv-python pillow

## preparando os caminhos

quando este notebook é aberto direto pelo github no google colab, os arquivos do repositório não aparecem automaticamente na máquina do colab.

por isso, a célula abaixo clona o repositório se a pasta do dataset ainda não existir.

In [ ]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/fezleep/image-recommendation-dio.git"
PROJECT_NAME = "image-recommendation-dio"

if "google.colab" in sys.modules:
    project_path = Path("/content") / PROJECT_NAME
    if not project_path.exists():
        !git clone -q {REPO_URL}
    os.chdir(project_path)
else:
    current = Path.cwd()
    if current.name == "notebooks":
        os.chdir(current.parent)

PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset"

sys.path.append(str(PROJECT_ROOT / "src"))

print("pasta do projeto:", PROJECT_ROOT)
print("dataset:", DATASET_DIR)

## importações

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from recommendation import (
    build_embeddings,
    list_images,
    load_feature_extractor,
    recommend_similar,
    show_recommendations,
)

## uma explicacao rapida sobre embeddings

um embedding é um vetor de números que resume uma imagem.

a mobilenetv2 já aprendeu muitos padrões visuais em uma base grande de imagens. aqui eu uso essa rede pronta para transformar cada produto em um vetor. depois, o projeto compara esses vetores para achar imagens parecidas.

## carregando o dataset

In [ ]:
image_paths = list_images(DATASET_DIR)

print(f"imagens encontradas: {len(image_paths)}")
for path in image_paths:
    print(path.relative_to(PROJECT_ROOT))

## visualizando algumas imagens

In [ ]:
from PIL import Image

plt.figure(figsize=(12, 6))

for index, path in enumerate(image_paths[:9], start=1):
    plt.subplot(3, 3, index)
    plt.imshow(Image.open(path))
    plt.title(path.parent.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

## carregando a mobilenetv2

a rede e carregada sem a camada final de classificacao. assim, ela funciona como extratora de caracteristicas.

In [ ]:
model = load_feature_extractor()
model.summary()

## extraindo os embeddings

In [ ]:
embeddings = build_embeddings(model, image_paths)

print("formato dos embeddings:", embeddings.shape)

## recomendando imagens parecidas

para testar, escolhi uma imagem do dataset como entrada. voce pode trocar o indice e rodar de novo.

In [ ]:
query_image = image_paths[0]

recommendations = recommend_similar(
    query_image=query_image,
    image_paths=image_paths,
    embeddings=embeddings,
    model=model,
    top_k=3,
)

print("imagem de entrada:", query_image.relative_to(PROJECT_ROOT))
print("recomendacoes:")
for item in recommendations:
    print(item["path"].relative_to(PROJECT_ROOT), "-", round(item["score"], 4))

## visualização final em grid

In [ ]:
show_recommendations(query_image, recommendations)

## conclusao

o resultado mostra um fluxo simples de recomendação visual.

a mobilenetv2 foi usada como extratora de características, os embeddings foram comparados com cosine similarity e as imagens mais próximas foram exibidas em um grid.

ainda dá para melhorar bastante, principalmente usando mais imagens reais e salvando os embeddings em arquivo, mas a base do sistema já está funcionando.